# AIDA 2154 Final Project – 03 Baseline Classifier and Missing-Item Flags

**Data version 2** (`shared/synthetic-data/kit-cards/`), regions from notebook 02's frozen segmentation.

**Question:** Using the regions found in notebook 02, can a transparent colour-and-shape classifier name each region as one of the five kit items or "not a kit item", and how well do the resulting missing-item flags work on validation cards?

This is the project's baseline. The coordinator's goal is to catch kits with an item left off, so the main measure is **missing-item recall**, reported together with missing-item precision and the false missing-item flags that create unnecessary checks. A more complex model (notebook 04) will be compared with this baseline on the same split, the same training regions and the same measures.

**Use of the splits.** The classifier's regularisation strength is chosen by cross-validation within the training cards, grouped by card, with feature scaling fitted inside each fold. Validation evaluates the candidate pipeline, but it remains development evidence because it previously informed segmentation in notebook 02. Test images are not opened; the complete frozen pipeline will be evaluated on them once, at the end of the project.

All cards are computer-generated and contain no personal information.

In [1]:
import hashlib
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from PIL import Image, ImageDraw
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
KIT = SD / "kit-cards"
OUTPUT_DIR = Path("../outputs")
FIGURE_DIR = Path("../figures")
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
SEED = 2026
ITEMS = ["access_badge", "safety_vest", "hard_hat", "laptop", "phone"]
NOT_KIT = "not_kit_item"
CLASSES = ITEMS + [NOT_KIT]
DEV_SPLITS = ["train", "validation"]  # only development images are read; metadata includes all splits

manifest = pd.read_csv(KIT / "manifest.csv", keep_default_na=False, na_values=[""])
objects = pd.read_csv(KIT / "icon_instances.csv", keep_default_na=False, na_values=[""])
settings = json.loads((OUTPUT_DIR / "segmentation_settings.json").read_text())      # frozen in notebook 02
regions_saved = pd.read_csv(OUTPUT_DIR / "regions_dev.csv", keep_default_na=False, na_values=[""])

# The frozen settings must still describe these exact inputs
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
dev_cards = manifest[manifest["split"].isin(DEV_SPLITS)].reset_index(drop=True)
dev_objects = objects[objects["split"].isin(DEV_SPLITS)].reset_index(drop=True)
image_records = [{"file_name": f, "sha256": sha(KIT / "images" / f)} for f in sorted(dev_cards["file_name"])]
recorded = settings["inputs_sha256"]
inputs_match = (recorded["manifest.csv"] == sha(KIT / "manifest.csv")
                and recorded["icon_instances.csv"] == sha(KIT / "icon_instances.csv")
                and recorded["generation_info.json"] == sha(KIT / "outputs/generation_info.json")
                and recorded["development_images (file names and contents, train + validation, sorted by name)"]
                == hashlib.sha256(json.dumps(image_records, sort_keys=True).encode()).hexdigest())
assert inputs_match, "The inputs no longer match the frozen segmentation settings from notebook 02"

# Saved region identities and card links
card_splits = manifest.set_index("card_id")["split"]
assert not regions_saved.duplicated(["card_id", "region"]).any(), "Duplicate saved region identifiers"
assert regions_saved["card_id"].isin(dev_cards["card_id"]).all(), "A saved region belongs to a non-development or unknown card"
assert regions_saved["split"].eq(regions_saved["card_id"].map(card_splits)).all(), "A saved region has the wrong split"

# The segmentation must be rebuilt with the same software versions it was frozen with
versions = settings["package_versions"]
assert cv2.__version__ == versions["opencv"], f"OpenCV {cv2.__version__} differs from the frozen {versions['opencv']}"
assert np.__version__ == versions["numpy"], f"NumPy {np.__version__} differs from the frozen {versions['numpy']}"

print(f"Data version {DATA_VERSION}; frozen segmentation: {settings['chosen']['name']}")
print(f"Development cards: {(dev_cards['split'] == 'train').sum()} training, {(dev_cards['split'] == 'validation').sum()} validation; "
      f"saved regions: {len(regions_saved)}")
print("Inputs match the frozen segmentation settings:", inputs_match)

Data version 2; frozen segmentation: masked | gray > 9 | close 11
Development cards: 360 training, 120 validation; saved regions: 1243
Inputs match the frozen segmentation settings: True


## A. Rebuilding each region's pixels

Notebook 02 saved each region's box, but colour and shape features need its **predicted foreground pixels** (the connected-component mask) rather than the entire box or the recorded object box. These masks may include noise or paper and may omit parts of a drawing. This section reruns the frozen segmentation exactly as recorded in `segmentation_settings.json` and keeps each region's mask.

As a provenance check, the rebuilt regions must match notebook 02's saved regions box for box, in region order, on every card. If they don't, the notebook stops, because the features would then describe different regions from the ones notebook 02 evaluated.

In [2]:
rows_a, rows_b = settings["content_area"]["rows"]           # inclusive bounds
cols_a, cols_b = settings["content_area"]["columns"]
CONTENT = (slice(rows_a, rows_b + 1), slice(cols_a, cols_b + 1))
chosen = settings["chosen"]
mb = settings["masked_background"]
assert chosen["background"] == "masked" and chosen["measure"] == "gray", "This notebook rebuilds the masked grayscale setting"
GRAY_WEIGHTS = np.array([0.114, 0.587, 0.299], dtype=np.float32)   # OpenCV loads images as blue-green-red
OPEN = int(settings["opening"].split(" x ")[0])


def segment(path):
    # The frozen pipeline; returns the denoised content (BGR), the paper estimate, the component label image and boxes
    image = cv2.imread(str(path))
    if image is None:
        raise ValueError(f"Could not open image: {path}")
    content = cv2.medianBlur(image, settings["median_filter"])[CONTENT].astype(np.float32)
    sigma = settings["background_sigma"]
    gaussian = cv2.GaussianBlur(content, (0, 0), sigma)
    prelim = np.linalg.norm(content - gaussian, axis=2) > mb["preliminary_colour_threshold"]
    d = int(mb["preliminary_mask_dilation"].split(" x ")[0])
    prelim = cv2.dilate(prelim.astype(np.uint8), np.ones((d, d), np.uint8)).astype(bool)
    remaining = (~prelim).astype(np.float32)
    weight = cv2.GaussianBlur(remaining, (0, 0), sigma)
    paper = cv2.GaussianBlur(content * remaining[..., None], (0, 0), sigma) / np.maximum(weight, mb["minimum_background_weight"])[..., None]
    paper = np.where(weight[..., None] > mb["minimum_background_weight"], paper, gaussian)
    diff = np.abs((content - paper) @ GRAY_WEIGHTS)
    mask = (diff > chosen["threshold"]).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((OPEN, OPEN), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (chosen["close"],) * 2))
    _, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=settings["connectivity"])
    keep = [k for k in range(1, len(stats)) if stats[k][4] >= settings["min_region_pixels"]]
    boxes = [(int(stats[k][0]) + CONTENT[1].start, int(stats[k][1]) + CONTENT[0].start,
              int(stats[k][0] + stats[k][2]) + CONTENT[1].start, int(stats[k][1] + stats[k][3]) + CONTENT[0].start) for k in keep]
    return content, paper, labels, keep, boxes


segmented = {}
mismatched_cards = []
for card in dev_cards.itertuples():
    content, paper, labels, keep, boxes = segment(KIT / "images" / card.file_name)
    saved_card = regions_saved[regions_saved["card_id"] == card.card_id].sort_values("region")
    assert saved_card["region"].tolist() == list(range(len(saved_card))), f"Unexpected region numbering on {card.card_id}"
    saved_boxes = list(saved_card[["x0", "y0", "x1", "y1"]].itertuples(index=False, name=None))
    if boxes != saved_boxes:
        mismatched_cards.append(card.card_id)
    segmented[card.card_id] = (content, paper, labels, keep)
regions_reproduced = not mismatched_cards
print(f"Rebuilt regions on {len(segmented)} cards; cards whose regions differ from notebook 02: {len(mismatched_cards)}")
assert regions_reproduced, f"Rebuilt regions differ from notebook 02 on {mismatched_cards[:5]}"

Rebuilt regions on 480 cards; cards whose regions differ from notebook 02: 0
